In [ ]:
import pandas as pd

# --- Charger les deux fichiers ---
df_weather = pd.read_csv("data/raw/weather_scored.csv")
df_hotels = pd.read_csv("data/raw/hotels.csv")

# --- Renommer les colonnes ambiguës AVANT la fusion ---
# Côté ville : lat/lon -> ville_lat/ville_lon
df_weather = df_weather.rename(columns={"lat": "ville_lat", "lon": "ville_lon"})

# Côté hôtel : lat/lon -> hotel_lat/hotel_lon, et note/nom explicites
df_hotels = df_hotels.rename(columns={
    "lat": "hotel_lat",
    "lon": "hotel_lon",
    "nom": "hotel_nom",
    "note": "hotel_note",
    "prix": "hotel_prix",
    "url": "hotel_url",
})

# --- LA FUSION sur city_id ---
# Chaque hôtel reçoit la météo de sa ville. Une ligne par hôtel.
df_enriched = df_hotels.merge(
    df_weather.drop(columns=["address_type"], errors="ignore"), 
    on="city_id",
    how="left",       
)

# --- On enlève la colonne 'ville' en double (elle vient des deux) si besoin ---
if "ville" in df_enriched.columns and "city" in df_enriched.columns:
    df_enriched = df_enriched.drop(columns=["ville"])

print("Lignes :", len(df_enriched))
print("Colonnes :", list(df_enriched.columns))
df_enriched.head()

Lignes : 875
Colonnes : ['hotel_nom', 'hotel_url', 'hotel_note', 'hotel_prix', 'hotel_lat', 'hotel_lon', 'city_id', 'city', 'ville_lat', 'ville_lon', 'temp_moy', 'clouds_moy', 'pop_moy', 'temp_score', 'ciel_score', 'pluie_score', 'score']


,hotel_nom,hotel_url,hotel_note,hotel_prix,hotel_lat,hotel_lon,city_id,city,ville_lat,ville_lon,temp_moy,clouds_moy,pop_moy,temp_score,ciel_score,pluie_score,score
0,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html,8.1,NaN,48.614700,-1.509617,1,Mont Saint Michel,48.635954,-1.51146,23.14125,91.375,0.9375,0.162751,0.061475,0.0,0.074742
1,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,8.1,NaN,48.617587,-1.510396,1,Mont Saint Michel,48.635954,-1.51146,23.14125,91.375,0.9375,0.162751,0.061475,0.0,0.074742
2,Le Saint Aubert,https://www.booking.com/hotel/fr/hotel-saint-a...,7.4,NaN,48.612938,-1.510105,1,Mont Saint Michel,48.635954,-1.51146,23.14125,91.375,0.9375,0.162751,0.061475,0.0,0.074742
3,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,8.4,NaN,48.614247,-1.510545,1,Mont Saint Michel,48.635954,-1.51146,23.14125,91.375,0.9375,0.162751,0.061475,0.0,0.074742
4,La Confiance,https://www.booking.com/hotel/fr/les-terrasses...,7.5,NaN,48.635300,-1.510397,1,Mont Saint Michel,48.635954,-1.51146,23.14125,91.375,0.9375,0.162751,0.061475,0.0,0.074742


In [4]:
# Sauvegardons dans un dossier transformés avant de deployer

from pathlib import Path

Path("data/curated").mkdir(parents=True, exist_ok=True)
output = Path("data/curated/kayak_enriched.csv")
df_enriched.to_csv(output, index=False)
print("Écrit :", output, "|", len(df_enriched), "lignes")

Écrit : data/curated/kayak_enriched.csv | 875 lignes


In [ ]:
import os
import boto3
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client("s3", region_name="eu-west-3")

BUCKET = os.getenv("S3_BUCKET")
BASE_PREFIX = os.getenv("S3_BASE_PREFIX")


def uploader_dossier(dossier_local, sous_prefixe):
    dossier = Path(dossier_local)
    fichiers = [f for f in dossier.rglob("*") if f.is_file()]

    for fichier in fichiers:
        chemin_relatif = str(fichier.relative_to(dossier)).replace("\\", "/")
        # .strip("/") sur chaque morceau
        cle_s3 = "/".join([
            BASE_PREFIX.strip("/"),
            sous_prefixe.strip("/"),
            chemin_relatif,
        ])
        s3.upload_file(str(fichier), BUCKET, cle_s3)
        print(f"  {cle_s3}")

    print(f"→ {len(fichiers)} fichiers uploadés\n")


# --- Upload ---
print("=== RAW ===")
uploader_dossier("data/raw", "raw")

print("=== CURATED ===")
uploader_dossier("data/curated", "curated")




=== RAW ===
  temp/tp-data-eng/jedha_certification_kayak/raw/cities.csv
  temp/tp-data-eng/jedha_certification_kayak/raw/weather_scored.csv
  temp/tp-data-eng/jedha_certification_kayak/raw/hotels.csv
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/23.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/29.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/15.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/2.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/6.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/26.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/17.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/5.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/11.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/30.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/7.json
  temp/tp-data-eng/jedha_certification_kayak/raw/weather/9.json
  temp/tp-data-eng/jedha_